In [25]:
!pip install khmer-nltk khmereasytools tqdm khmernormalizer jiwer transformers datasets peft evaluate accelerate -q

In [33]:
# dependencies
import re
import math
from collections import Counter, defaultdict
import torch
import numpy as np
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient

# khmer specific
from khmernltk import word_tokenize
from khmernormalizer import normalize

# hugging Face ecosystem
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForMaskedLM, 
    DataCollatorForLanguageModeling, 
    TrainingArguments, 
    Trainer, 
    TrainerCallback
)
from peft import LoraConfig, get_peft_model

In [13]:
# dict_path = '/kaggle/input/datasets/sopheakchan/spell-checker-elements/khmer_dictionary.txt'

# def load_dictionary(path: str) -> set:
#     dictionary = set()
#     with open(path, 'r', encoding='utf-8') as f:
#         for line in f:
#             word = line.strip()
#             if word:
#                 dictionary.add(word)
#     print(f"Dictionary: {len(dictionary):,} words")
#     return dictionary

# dictionary = load_dictionary(dict_path)

# # Quick check
# for test in ['ក', 'កម', 'កមព', 'ជា', 'កមពជា']:
#     print(f"  '{test}': {test in dictionary}")

# Load the data from Hugging face

In [11]:
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# Load already cleaned sentence data
clean_ds = load_dataset(
    "sopheakchan/khmer_sentence_data",
    split="train",
    token=hf_token
)

sentences = clean_ds["text"]

In [12]:
# Quick check
print(f"Total sentences: {len(sentences)}")
print("\nSample:")
for i, s in enumerate(sentences[:10]):
    words = [w for w in word_tokenize(s, return_tokens=True)
             if w.strip() and w != ' ']
    print(f"  [{i+1:02d}] {len(words):2d}w  {s}")

Total sentences: 1260369

Sample:
  [01] 10w  កណ្តាលៈអ្នកដែលមានគោលបំណងបំផ្លិចបំផ្លាញសុខសន្តិភាពគឺតែងតែ
  [02] 10w  ញុះញង់ដោយអំពាវនាវឱ្យកងកម្លាំងប្រដាប់អាវុធឱ្យចូលរួមដើម្បីផ្តួលរំលំរដ្ឋាភិបាល
  [03] 10w  ប៉ុន្តែនៅគ្រប់ប្រទេសនៅលើពិភពលោកការធ្វើរដ្ឋប្រហារផ្តួលរំលំរដ្ឋាភិបាល
  [04] 10w  កម្លាំងប្រដាប់អាវុធគឺជាការខុសច្បាប់មិនថាប្រទេសប្រជាធិបតេយ្យឬប្រទេស
  [05] 10w  ណានោះទេគ្មានអ្វីដែលនាំឱ្យមានសេចក្តីសុខដល់
  [06] 10w  ប្រទេសជាតិយើងតាមរយៈការវាយប្រហារគ្នានោះទេមិនចាំបាច់មើល
  [07] 10w  ដល់៤០ឆ្នាំមុនទេមើលតែប្រទេសមួយចំនួននៅ
  [08] 10w  មជ្ឈិមបូព៌ាសព្វថ្ងៃតើមានប្រទេសណាមួយដែលមានសេចក្តីសុខ
  [09]  9w  នេះប្រសាសន៍លើកឡើងរបស់ឧត្តមសេនីយ៍ឯកហ៊ុនម៉ាណែតអគ្គមេបញ្ជាការ
  [10] 10w  រងកងយោធពលខេមរភូមិន្ទនិងជាមេបញ្ជាការកងទ័ពជើងគោកថ្លែងនៅក្នុង


In [29]:
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Confirm we have 2 GPUs — script will still work with 1
assert torch.cuda.device_count() >= 1, "No GPU found!"

CUDA available  : True
GPU count       : 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


# Load the model

In [34]:
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [35]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Tokenizing dataset for XLM-RoBERTa...")
tokenized_datasets = clean_ds.map(tokenize_function, batched=True, remove_columns=["text"])

Tokenizing dataset for XLM-RoBERTa...


Map:   0%|          | 0/1260369 [00:00<?, ? examples/s]

# Data splitting

In [36]:
# Split 1: Separate 70% for training, 30% for Validation+Test
train_remainder = tokenized_datasets.train_test_split(test_size=0.30, seed=42)
train_dataset = train_remainder["train"]

# Split 2: Divide the remaining 30% into 20% Validation and 10% Test
val_test = train_remainder["test"].train_test_split(test_size=1/3.0, seed=42)
eval_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)} | Test: {len(test_dataset)}")

Train: 882258 | Eval: 252074 | Test: 126037


# Custome callback

In [37]:
class ShowPredictionsCallback(TrainerCallback):
    def __init__(self, tokenizer, eval_dataset, num_samples=3):
        self.tokenizer = tokenizer
        self.sample_dataset = eval_dataset.select(range(num_samples))
        self.collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

    def on_evaluate(self, args, state, control, model, **kwargs):
        print("\n" + "="*60)
        print(f"Visualizing Predictions at Step {state.global_step}")
        print("="*60)
        
        model.eval()
        features = [self.sample_dataset[i] for i in range(len(self.sample_dataset))]
        batch = self.collator(features)
        
        device = next(model.parameters()).device
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits = outputs.logits
        
        predictions = torch.argmax(logits, dim=-1)
        
        for i in range(len(features)):
            print(f"\nSample {i+1}:")
            current_inputs = input_ids[i].cpu().numpy()
            current_labels = labels[i].cpu().numpy()
            current_preds = predictions[i].cpu().numpy()
            
            # Find masked indices where labels are not -100
            masked_indices = np.where(current_labels != -100)[0]
            
            if len(masked_indices) == 0:
                print("  [No tokens masked in this sample]")
                continue
                
            # Reconstruct what the model received as input by showing <mask> tokens
            masked_sentence = self.tokenizer.decode(current_inputs, skip_special_tokens=False)
            # Clean up the output string formatting slightly for easier viewing
            masked_sentence = masked_sentence.replace("<s>", "").replace("</s>", "").strip()
            
            print(f"  [Input Context]: {masked_sentence}")
            
            for idx in masked_indices:
                true_token = self.tokenizer.decode([current_labels[idx]])
                pred_token = self.tokenizer.decode([current_preds[idx]])
                
                # Check if it was correct
                status = "MATCH" if true_token.strip() == pred_token.strip() else "MISMATCH"
                print(f"    -> Position [{idx:2d}]: True: '{true_token}' | Predicted: '{pred_token}' ({status})")
        print("="*60 + "\n")

# Model and Lora config

In [38]:
print("Loading Base Model and applying LoRA...")
base_model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading Base Model and applying LoRA...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 589,824 || all params: 278,885,010 || trainable%: 0.2115


In [40]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

# Training setup

In [41]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/xlm-roberta-khmer-lora",
    eval_strategy="steps",
    max_steps=200,                    
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    per_device_train_batch_size=32,   # Optimized for 16GB T4 GPUs
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    fp16=True,                      
    report_to="none"
)

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[ShowPredictionsCallback(tokenizer, eval_dataset, num_samples=3)]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
50,10.186241,4.686152
100,10.344820,4.583117
150,9.956630,4.514557
200,9.914320,4.533719



Visualizing Predictions at Step 50

Sample 1:
  -> Pos [18]: True: 'ម្តង' | Predicted: 'ប៉ុណ្ណោះ'

Sample 2:
  -> Pos [6]: True: 'ហើយ' | Predicted: 'ហើយ'
  -> Pos [8]: True: 'សង្ឃឹម' | Predicted: 'អះអាង'
  -> Pos [11]: True: 'នេះ' | Predicted: 'បាន'

Sample 3:
  -> Pos [7]: True: 'ថា' | Predicted: 'ថា'



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Visualizing Predictions at Step 100

Sample 1:
  -> Pos [8]: True: 'ការ' | Predicted: 'ការ'
  -> Pos [10]: True: 'អ' | Predicted: 'អ'
  -> Pos [19]: True: 'ទី' | Predicted: 'ទី'

Sample 2:
  -> Pos [8]: True: 'សង្ឃឹម' | Predicted: 'បាន'
  -> Pos [9]: True: 'ថា' | Predicted: 'ថា'

Sample 3:
  -> Pos [3]: True: 'តុលាការ' | Predicted: 'បាន'
  -> Pos [4]: True: 'ព' | Predicted: 'ព'
  -> Pos [7]: True: 'ថា' | Predicted: 'ថា'
  -> Pos [10]: True: 'ជា' | Predicted: 'ជា'



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Visualizing Predictions at Step 150

Sample 1:
  -> Pos [11]: True: 'ត្ថ' | Predicted: 'ត្ថ'

Sample 2:
  [No tokens masked in this sample]

Sample 3:
  -> Pos [6]: True: 'យល់' | Predicted: 'យល់'
  -> Pos [8]: True: 'មិន' | Predicted: 'ការ'
  -> Pos [9]: True: 'មែន' | Predicted: 'នេះ'
  -> Pos [11]: True: 'អា' | Predicted: 'អា'
  -> Pos [13]: True: 'ភាព' | Predicted: 'ភាព'
  -> Pos [14]: True: 'គឺ' | Predicted: 'ភាព'



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Visualizing Predictions at Step 200

Sample 1:
  -> Pos [12]: True: 'ប្រយោជន៍' | Predicted: 'ប្រយោជន៍'
  -> Pos [14]: True: 'ទឹកដី' | Predicted: 'នាម'
  -> Pos [18]: True: 'ម្តង' | Predicted: 'ម្តង'
  -> Pos [22]: True: 'កម្ពុជា' | Predicted: 'ចិន'

Sample 2:
  -> Pos [1]: True: 'នេះ' | Predicted: 'នេះ'
  -> Pos [4]: True: 'យូរ' | Predicted: 'រួច'
  -> Pos [9]: True: 'ថា' | Predicted: 'ថា'

Sample 3:
  -> Pos [7]: True: 'ថា' | Predicted: 'ថា'
  -> Pos [13]: True: 'ភាព' | Predicted: 'ភាព'



TrainOutput(global_step=200, training_loss=10.343609962463379, metrics={'train_runtime': 7446.8384, 'train_samples_per_second': 3.438, 'train_steps_per_second': 0.027, 'total_flos': 1168746868304640.0, 'train_loss': 10.343609962463379, 'epoch': 0.029014942695488176})

# Evaluate

In [44]:
test_results = trainer.evaluate(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Visualizing Predictions at Step 200

Sample 1:
  [No tokens masked in this sample]

Sample 2:
  -> Pos [3]: True: 'ធ្វើឡើង' | Predicted: 'ធ្វើឡើង'

Sample 3:
  -> Pos [12]: True: 'ទិ' | Predicted: 'ទិ'



In [45]:
# Calculate MLM Perplexity
test_loss = test_results.get("eval_loss")
if test_loss is not None:
    perplexity = math.exp(test_loss)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Perplexity: {perplexity:.4f}")
else:
    print("Could not calculate test loss.")

Test Loss: 4.5112
Test Perplexity: 91.0332


In [47]:
model.eval()
num_test_samples = 5 
sample_test_data = test_dataset.select(range(num_test_samples))

# Use the data collator to dynamically mask 15% of the tokens
features = [sample_test_data[i] for i in range(len(sample_test_data))]
batch = data_collator(features)

# Move inputs to the active device (GPU)
device = next(model.parameters()).device
input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    outputs = model(input_ids=input_ids)
    logits = outputs.logits

# Get the highest-scoring token ID for every position
predictions = torch.argmax(logits, dim=-1)

for i in range(len(features)):
    print(f"\nTest Sample {i+1}:")
    current_inputs = input_ids[i].cpu().numpy()
    current_labels = labels[i].cpu().numpy()
    current_preds = predictions[i].cpu().numpy()
    
    # Locate where masks were placed (where labels are not -100)
    masked_indices = np.where(current_labels != -100)[0]
    
    if len(masked_indices) == 0:
        print("  [No tokens were randomly masked in this sentence slice]")
        continue
        
    # Reconstruct what the model was fed (with <mask> strings visible)
    reconstructed_input = []
    for idx, token_id in enumerate(current_inputs):
        if current_labels[idx] != -100:
            reconstructed_input.append(tokenizer.mask_token)
        else:
            reconstructed_input.append(tokenizer.decode([token_id]))
    
    print(f"  Input Sent: {''.join(reconstructed_input).strip()}")
    
    # Print the true vs predicted candidates for each mask
    for idx in masked_indices:
        true_word = tokenizer.decode([current_labels[idx]])
        pred_word = tokenizer.decode([current_preds[idx]])
        
        match_status = "MATCH" if true_word.strip() == pred_word.strip() else "MISMATCH"
        print(f"  -> Position [{idx}]: True: '{true_word}' | Predicted: '{pred_word}' ({match_status})")


Test Sample 1:
  Input Sent: <s><mask>ឆ្នាំ២០១៩ស្អែកនេះសូម<mask>បញ្ជាក់ថាតាមរយៈហិរ<mask>ប្បទានឥត</s>
  -> Position [1]: True: 'ខែតុលា' | Predicted: 'ដើម' (MISMATCH)
  -> Position [10]: True: 'ុំ' | Predicted: 'បញ្ជាក់' (MISMATCH)
  -> Position [16]: True: 'ញ្ញ' | Predicted: 'ញ្ញ' (MATCH)

Test Sample 2:
  Input Sent: <s>ចំណាយ<mask><mask>ការការពារប្រទេសជំនួយម្ហូបអាហារគ្រាអា<mask><mask>ន</s>
  -> Position [3]: True: 'លើ' | Predicted: 'លើ' (MATCH)
  -> Position [4]: True: 'កិច្ច' | Predicted: 'កិច្ច' (MATCH)
  -> Position [17]: True: 'ស' | Predicted: 'ស' (MATCH)
  -> Position [18]: True: 'ន្' | Predicted: 'ន្' (MATCH)

Test Sample 3:
  Input Sent: <s>ដែលបានអនុម<mask>តនៅក្នុងសម័យប្រជុំបច្ចេកទេសទី៣៦និង</s><pad><pad><pad><pad><pad>
  -> Position [5]: True: '័' | Predicted: '័' (MATCH)

Test Sample 4:
  Input Sent: <s>ផ្ទុះបីគ្រាប់នៅទីក្រុងម<mask>បៃកាលពីខែមុន</s><pad><pad><pad><pad><pad><pad><pad>
  -> Position [7]: True: 'ុម' | Predicted: 'ូ' (MISMATCH)

Test Sample 5:
  Input Sent: <s>នេះគ

In [48]:
# Save final adapters
output_path = "/kaggle/working/khmer-spell-adapter-final"
trainer.model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"LoRA adapter and tokenizer saved successfully to {output_path}.")

LoRA adapter and tokenizer saved successfully to /kaggle/working/khmer-spell-adapter-final.


# Inference

In [50]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import PeftModel

# Define paths
model_checkpoint = "xlm-roberta-base"
adapter_path = "/kaggle/working/khmer-spell-adapter-final" # Or wherever you saved it

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

# Attach your trained LoRA adapter to the base model
print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() # Set to evaluation mode

# Test it with a masked sentence
text = "ខ្ញុំចូលចិត្តញ៉ាំ <mask> នៅពេលព្រឹក។" 
inputs = tokenizer(text, return_tensors="pt").to(device)

print(f"\nOriginal text: {text}")

with torch.no_grad():
    outputs = model(**inputs)

# Find where the <mask> token is in the sentence
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

if len(mask_token_index) > 0:
    # Extract raw logits for the masked position
    mask_logits = outputs.logits[0, mask_token_index, :] # Shape: [1, vocab_size]
    
    # Convert logits into probabilities using Softmax
    probabilities = F.softmax(mask_logits, dim=-1)
    
    # Extract the top 5 highest probabilities and their corresponding token IDs
    top_5_results = torch.topk(probabilities, 5, dim=1)
    top_5_probs = top_5_results.values[0].tolist()
    top_5_tokens = top_5_results.indices[0].tolist()

    print("\nTop 5 Predictions with Confidence:")
    print("-" * 40)
    for rank, (token_id, prob) in enumerate(zip(top_5_tokens, top_5_probs), 1):
        word = tokenizer.decode([token_id])
        # Format the probability to show as a percentage with 2 decimal places
        print(f"Top {rank}: '{word}' | Confidence: {prob * 100:.2f}%")
    print("-" * 40)
else:
    print("No <mask> token found in the input string.")

Loading tokenizer and base model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attaching LoRA adapter...

Original text: ខ្ញុំចូលចិត្តញ៉ាំ <mask> នៅពេលព្រឹក។

Top 5 Predictions with Confidence:
----------------------------------------
Top 1: 'ស្រា' | Confidence: 17.19%
Top 2: 'បាយ' | Confidence: 16.29%
Top 3: 'ទឹក' | Confidence: 11.78%
Top 4: 'អាហារ' | Confidence: 10.99%
Top 5: 'សាច់' | Confidence: 10.79%
----------------------------------------


# Take multiple function

In [51]:
import torch
import torch.nn.functional as F

def predict_masked_sentences(sentences, model, tokenizer, top_k=3):
    """
    Takes a list of sentences (containing <mask>), 
    and prints the top predictions with confidence scores.
    """
    for i, text in enumerate(sentences, 1):
        print(f"Sample {i}: {text}")
        
        # Tokenize and send to the same device as the model (GPU/CPU)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Locate the mask token in the sequence
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        if len(mask_token_index) == 0:
            print("  [Error: No <mask> token found in this sentence. Please include <mask>]")
            print("-" * 50)
            continue
            
        # For this function, we evaluate the first mask found in the sentence
        first_mask_loc = mask_token_index[0]
        mask_logits = outputs.logits[0, first_mask_loc, :]
        
        # Convert to probabilities
        probabilities = F.softmax(mask_logits, dim=-1)
        top_results = torch.topk(probabilities, top_k, dim=-1)
        
        top_probs = top_results.values.tolist()
        top_tokens = top_results.indices.tolist()
        
        # Print the predictions
        for rank, (token_id, prob) in enumerate(zip(top_tokens, top_probs), 1):
            word = tokenizer.decode([token_id])
            print(f"  {rank}. '{word}' | {prob * 100:.2f}%")

In [55]:
my_test_sentences = [
    "អ្នកចេះនិយាយភាសា<mask>ទេ",
    "ខ្ញុំស្រលាញ់ប្រទេស<mask>ជា",
    "សេដ្ឋកិច្ច<mask>កំពុងអភិវឌ្ឍ",
    "យើងរស់នៅក្នុង<mask>មួយ"
]

predict_masked_sentences(my_test_sentences, model, tokenizer, top_k=4)

Sample 1: អ្នកចេះនិយាយភាសា<mask>ទេ
  1. 'ខ្មែរ' | 42.91%
  2. 'អង់គ្លេស' | 11.86%
  3. 'ចិន' | 11.50%
  4. 'បរទេស' | 7.85%
Sample 2: ខ្ញុំស្រលាញ់ប្រទេស<mask>ជា
  1. 'ខ្ញុំ' | 34.79%
  2. 'ជាតិ' | 11.54%
  3. 'ជប៉ុន' | 7.14%
  4. 'បារាំង' | 6.94%
Sample 3: សេដ្ឋកិច្ច<mask>កំពុងអភិវឌ្ឍ
  1. 'និង' | 49.90%
  2. 'និង' | 25.87%
  3. 'កម្ពុជា' | 4.03%
  4. 'ជាតិ' | 3.24%
Sample 4: យើងរស់នៅក្នុង<mask>មួយ
  1. 'ទីក្រុង' | 23.60%
  2. 'ប្រទេស' | 12.57%
  3. 'តំបន់' | 11.96%
  4. 'គ្រួសារ' | 7.51%
